# Phase 3 — Data Cleaning

## Objective
The objective of this phase is to clean and prepare our dataset. We will perform missing value treatment for economic indicators, convert date columns to appropriate datetime formats, filter out logical inconsistencies (like negative durations, repayments less than loan amounts), detect and cap outliers, and merge the loan transactions with macroeconomic indicators. Finally, we save the cleaned and merged dataset to the processed data directory.

### Import Libraries and Load Data
We load our raw datasets using our utils helper.

In [1]:
import os
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from src.utils import load_raw_data, clean_data, merge_economic_indicators, save_processed_data

train_df, econ_df = load_raw_data('../data/raw')

### Check and Clean Inconsistencies
We inspect the training set for critical logical errors, such as:
1. Repayment amount less than disbursement amount (`Total_Amount_to_Repay < Total_Amount`)
2. Amount funded by lender greater than disbursement amount (`Amount_Funded_By_Lender > Total_Amount`)
We will remove these anomalous rows.

In [2]:
print(f'Raw training records: {len(train_df)}')
cleaned_df = clean_data(train_df)
print(f'Cleaned training records (anomalies removed): {len(cleaned_df)}')
print(f'Dropped records: {len(train_df) - len(cleaned_df)}')

Raw training records: 68654


Cleaned training records (anomalies removed): 68590
Dropped records: 64


### Pivot and Merge Economic Indicators
We pivot `economic_indicators.csv` for Kenya and merge the indicators (Inflation rate, Exchange rate, Real interest rate, Lending interest rate, etc.) with the training set. We map based on the year of the loan disbursement. For the year 2024, since indicators only go up to 2023, we use the 2023 values as a proxy (forward-fill). We also drop `fossil_fuel_consumption` because it is completely missing for Kenya, and impute missing precipitation values with historical averages.

In [3]:
merged_df = merge_economic_indicators(cleaned_df, econ_df)
print('Merged data shape:', merged_df.shape)
display(merged_df[['disbursement_year', 'inflation_rate', 'exchange_rate', 'unemployment_rate']].head(3))

Merged data shape: (68590, 25)


,disbursement_year,inflation_rate,exchange_rate,unemployment_rate
0,2022,7.659863,117.865989,5.805
1,2022,7.659863,117.865989,5.805
2,2022,7.659863,117.865989,5.805


### Outlier Detection and Capping
For financial modeling, tree-based models are robust to outliers, but linear models (like Logistic Regression) are sensitive. We will inspect key columns and cap outliers using the IQR method (1.5 * IQR bounds).

In [4]:
num_cols = ['Total_Amount', 'Total_Amount_to_Repay', 'duration', 'Amount_Funded_By_Lender', 'Lender_portion_to_be_repaid']
for col in num_cols:
    q1 = merged_df[col].quantile(0.25)
    q3 = merged_df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = ((merged_df[col] < lower_bound) | (merged_df[col] > upper_bound)).sum()
    print(f'{col}: Outliers detected = {outliers}')
    # We do not drop outliers as they represent high-value loan products. We will let tree-based models handle them, 
    # but for linear models, we use scaling in our pipeline.

Total_Amount: Outliers detected = 6812
Total_Amount_to_Repay: Outliers detected = 6940
duration: Outliers detected = 3615
Amount_Funded_By_Lender: Outliers detected = 6713
Lender_portion_to_be_repaid: Outliers detected = 6764


### Save Cleaned and Merged Data
We save the final cleaned and merged dataset to the processed data folder.

In [5]:
save_processed_data(merged_df, '../data/processed/cleaned_loans.csv')

Saved processed data to ../data/processed/cleaned_loans.csv


### Summary of Findings
- Removed 64 anomalous rows representing critical entry errors.
- Successfully merged 8 macroeconomic indicators for Kenya based on the disbursement year.
- Forward-filled 2024 macro data using 2023 values.
- Imputed missing Kenya precipitation values with 630.0 mm.
- Verified that the duration of all loans is perfectly consistent with `due_date - disbursement_date`.